## Copying repo from git (FOR GOOGLE COLAB ONLY!!!)

In [ ]:
import os
import sys

!git clone https://github.com/isachagit/ML-DS-pet-projects/JerryDetect

%cd JerryDetect

sys.path.append(os.getcwd())

!nvidia-smi

## Lib Installation & Import

In [21]:
import os
import subprocess
import sys


def install_requirements(file_path="requirements.txt"):
    if not os.path.exists(file_path):
        print("Missing", file_path)
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", file_path])
        print("Successfully installed")
    except subprocess.CalledProcessError as e:
        print(f"Error: {e}")

install_requirements()

In [22]:
import numpy as np
import re
import os
import matplotlib.pyplot as plt
from typing import Text
import torch
from torch.utils.data import BatchSampler, SequentialSampler, Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("device:", device)

device: cpu


# <center>LETTER PREDICTION </center>

## Custom dataset

In [52]:
text = ''
with open(r'train_data/fragment_cat.txt', 'r', encoding='utf-8-sig') as f:
    for line in f:
        line = line.rstrip()
        line = line.replace('.', '')
        line = line.replace('’', '')
        line = line.replace('‘', '')
        line = line.replace(',', '')
        line = line.replace(':', '')
        line += ' '
        text += line
text = text.lower()
words = text.split()
chars = sorted(list(set(text)))
print('vocabulary:', chars)
char_to_index = {ch:i for i, ch in enumerate(chars)}
index_to_char = {i:ch for i, ch in enumerate(chars)}
voc_len = len(chars)
print("vocabulary length:", voc_len, '\n')

def word_to_tensor(word):
  tensor = torch.zeros(len(word), voc_len)
  for i, ch in enumerate(word):
    tensor[i, char_to_index[ch]] = 1.0
  return tensor
def word_to_target(word):
  targets = [char_to_index[word[i]] for i in range(1, len(word))]
  return torch.LongTensor(targets)

class OneHotDataset(Dataset):
    def __init__(self, text, seq_len=3, test=False):
        self.X = []
        self.y = []
        words = re.findall(r'\w+\s*', text)
        if test==True:
            for i in words:
                self.X.append(word_to_tensor(i))
        else:
            for word in words:
                if len(word)<=2 or seq_len>=len(word):
                    continue
                for i in range(len(word)-seq_len):
                    self.X.append(word_to_tensor(word[i:i+seq_len]))
                    self.y.append(word_to_tensor(word[i+seq_len]))
    def __len__(self):
        return len(self.X)
    def __getitem__(self, index):
        return self.X[index], self.y[index]


dataset_train = OneHotDataset(text, 3)
print('train words:', len(dataset_train))
train_loader = DataLoader(dataset_train, batch_size=1, shuffle=True)

letters_test = ''
with open(r'test_data/cat_letters.txt', 'r', encoding='utf-8-sig') as f:
    for line in f:
        letters_test += line
letters_test += ' '
dataset_test = OneHotDataset(letters_test, test=True)

test_loader = DataLoader(dataset_test, batch_size=1, shuffle=False)
print("test words:", len(dataset_test))

vocabulary: [' ', '!', '-', ';', '?', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', '—']
vocabulary length: 31 

train words: 3717
test words: 8


## Model init & Train loop

In [4]:
class OneHotLSTM(nn.Module):
    def __init__(self, voc_len, hidden_size, num_layers):
        super().__init__()
        self.voc_len = voc_len
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(voc_len, hidden_size, num_layers = num_layers, batch_first = True, dropout=0.3) 
        self.fc = nn.Linear(hidden_size, voc_len)
    def forward(self, x):
        out, (ht, ct) = self.lstm(x)
        out = out[:, -1, :]
        logits = self.fc(out)
        return logits

EPOCHS = 200

model = OneHotLSTM(voc_len, voc_len*2, 2).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

model.train()

for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)
        pred = model(batch_X)
        loss = criterion(pred.squeeze(0), batch_y.squeeze(0).squeeze(0))
        epoch_loss += loss.item()
        loss.backward()
        optimizer.step()
    print('epoch', epoch, 'loss: ', epoch_loss)


epoch 0 loss:  1307.1218112984207
epoch 1 loss:  1192.9060440969188
epoch 2 loss:  1171.9950220442843
epoch 3 loss:  1151.1746364280261
epoch 4 loss:  1106.9416152173362
epoch 5 loss:  1039.9149958080452
epoch 6 loss:  969.137303302472
epoch 7 loss:  912.9160352064937
epoch 8 loss:  854.1071706933253
epoch 9 loss:  818.8754372492222
epoch 10 loss:  739.2440761620601
epoch 11 loss:  706.9031894010623
epoch 12 loss:  663.240657616273
epoch 13 loss:  614.4155418400679
epoch 14 loss:  610.6131305369963
epoch 15 loss:  559.8707781254707
epoch 16 loss:  524.3095930993013
epoch 17 loss:  478.2706039800032
epoch 18 loss:  441.3896218652844
epoch 19 loss:  461.35244680070537
epoch 20 loss:  447.8604210587125
epoch 21 loss:  398.85589610248735
epoch 22 loss:  415.45537091120013
epoch 23 loss:  384.2805494373732
epoch 24 loss:  353.4959320356079
epoch 25 loss:  313.54597039270084
epoch 26 loss:  290.2715574657659
epoch 27 loss:  255.8717888724923
epoch 28 loss:  236.27294196840205
epoch 29 loss: 

## Testing

In [5]:
model.eval()

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X = batch_X.to(device)
        pred = model(batch_X).squeeze(0)
        pred = pred.cpu()
        batch_X = batch_X.cpu()

        index_voc = np.argmax(pred.numpy())

        for i in range(batch_X.squeeze(0).shape[0]):
            print(index_to_char[np.argmax(batch_X.squeeze(0)[i].numpy())], end='')

        while index_to_char[index_voc]!=' ':

          index_voc = np.argmax(pred.numpy())
          batch_X_mod = batch_X.squeeze(0)

          next_char_tensor = torch.zeros(1, voc_len)
          next_char_tensor[0, index_voc] = 1.0
          batch_X_mod = torch.cat((batch_X_mod[1:,:], next_char_tensor), dim=0)
          batch_X = batch_X_mod.unsqueeze(0)

          print('\033[31m', index_to_char[index_voc], '\033[0m', sep='', end='')
          batch_X = batch_X.to(device)
          pred = model(batch_X).squeeze(0)
          pred = pred.cpu()
          batch_X = batch_X.cpu()


поговорили вообще родителями пробовать бультерьер 

# <center> WORD PREDICTION </center>

In [53]:
text = ''
with open(r'train_data/fragment_cat.txt', 'r', encoding='utf-8-sig') as f:
    for line in f:
        line = line.rstrip()
        line = line.replace('.', ' .')
        line = line.replace('’', ' ’')
        line = line.replace('‘', '‘ ')
        line = line.replace(',', ' ,')
        line = line.replace(':', ' : ')
        line += ' '
        text += line
text = text.lower()
words = text.split()
voc_words = sorted(list(set(words)))
print(voc_words)
voc_words_size = len(voc_words)
word_to_idx = {w:i for i, w in enumerate(voc_words)}
idx_to_word = {i:w for i, w in enumerate(voc_words)}
print("vocabulary length:", '\033[32m', voc_words_size, '\033[0m', "words")
class WordDataset(Dataset):
  def __init__(self, words, context_size, test=False):
    words = words.split()
    self.X = []
    self.y = []
    #print(words)
    if test==True:
        for i in range(len(words) - context_size+1):
          self.X.append(list(word_to_idx[words[j]] for j in range(i, i+context_size)))
          self.y.append(word_to_idx[words[i+context_size-1]])
    else:
        for i in range(len(words) - context_size):
          self.X.append(list(word_to_idx[words[j]] for j in range(i, i+context_size)))
          self.y.append(word_to_idx[words[i+context_size]])
    self.X = torch.tensor(self.X, dtype=torch.long)
    self.y = torch.tensor(self.y, dtype=torch.long)
      
  def __len__(self):
    return len(self.y)
  def __getitem__(self, index):
     return self.X[index], self.y[index]


dataset_words_train = WordDataset(text, 3)

words_train_loader = DataLoader(dataset_words_train, batch_size=1, shuffle=True)

[',', '.', ':', 'a', 'about', 'afraid', 'after', 'again', 'against', 'ah', 'ah!', 'alike', 'all', 'along', 'always', 'am', 'and', 'angry', 'animals', 'another', 'any', 'anybody', 'anything', 'arched', 'are', 'armfuls', 'arms', 'as', 'asked', 'at', 'away', 'baby', 'back', 'bargain', 'bat', 'bat—that', 'bat—the', 'be', 'beautiful', 'beautifully', 'because', 'before', 'before;', 'began', 'behold!—the', 'being', 'beloved', 'bend', 'bent', 'beside', 'best', 'big', 'black', 'blade-bone', 'blade-bone—and', 'blessing', 'bone', 'bones', 'braided', 'breathed', 'broad', 'busy', 'but', 'by', 'came', 'can', 'care', 'carried', 'carry', 'cat', 'catch', 'cave', 'cave?', 'cheek', 'chin', 'clever', 'come', 'comes', 'comfy', 'cooed;', 'cooking', 'could', 'cow', 'crawled', 'cried', 'cruel', 'crying', 'curtain', 'cut', 'day', 'delicious', 'did', 'do', 'dog', 'dog?', 'doing', 'done', 'down—whoosh!—because', 'drag', 'dried', 'drink', 'eat', 'either', 'end;', 'enemy', 'even', 'evening', 'ever', 'every', 'ever

In [39]:
from IPython.display import clear_output
class WordLSTM(nn.Module):
  def __init__(self, voc_words_size, embedding_size, hidden_size, num_layers):
    super().__init__()
    self.embedding = nn.Embedding(voc_words_size, embedding_size)
    self.lstm = nn.LSTM(embedding_size, hidden_size, num_layers, batch_first=True)
    self.fc = nn.Linear(hidden_size, voc_words_size)
  def forward(self, x, hidden=None):
    out = self.embedding(x)
    out, hidden = self.lstm(out)
    out = self.fc(out[:, -1, :])
    return out, hidden

EMBEDDING_SIZE = 64

model = WordLSTM(voc_words_size, EMBEDDING_SIZE, EMBEDDING_SIZE*2, 3).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.005)
criterion = nn.CrossEntropyLoss()
EPOCHS = 90
for epoch in range(EPOCHS):
  epoch_loss = 0.0
  for batch_X, batch_y in words_train_loader:
    batch_X.to(device)
    batch_y.to(device)
    optimizer.zero_grad()
    pred, _ = model(batch_X)
    loss = criterion(pred, batch_y)
    loss.backward()
    optimizer.step()
    epoch_loss += loss.item()
  clear_output(wait=True)
  print('epoch', epoch, ':', epoch_loss/len(train_loader))


epoch 89 : 0.35539928026582845


In [41]:
words_dataset_test = WordDataset("test_data/", 3, test=True)
words_test_loader = DataLoader(words_dataset_test, batch_size=1)

model.eval()
with torch.no_grad():
    for batch_X, batch_y in words_test_loader:
        batch_x = batch_X
        for i in batch_X:
            print(idx_to_word[i.item()],end='')
        for i in range(2):
            pred, _ = model(batch_x)
            print('\033[32m', idx_to_word[torch.max(pred, dim=1)[1].item()], '\033[0m', end='')
            batch_x = torch.max(pred, dim=1)[1].unsqueeze(0)
        print('\n')

я главное  нет  невозможного  .  рождаются  ,  как  позитивность  .  рождаются  ,  как  позитивность  .  рождаются  ,  как  позитивность  .  рождаются 

сердце ,  как  позитивность  .  рождаются  ,  как  позитивность  .  рождаются  ,  как  позитивность  .  рождаются  ,  как  позитивность  .  рождаются 

супергерой .  рождаются  ,  как  позитивность  .  рождаются  ,  как  позитивность  .  рождаются  ,  как  позитивность  .  рождаются  ,  как  позитивность 



In [94]:
text = ''
with open(r'C:\\Users\user\Downloads\train_data_true.txt', 'r', encoding='utf-8-sig') as f:
    for line in f:
        line = line.rstrip()
        line = line.replace('.', ' .')
        line = line.replace(',', ' ,')
        line = line.replace(':', ' :')
        line += ' '
        text += line
text = text.lower()
words = text.split()
voc_words = sorted(list(set(words)))
#print(voc_words)
voc_words_size = len(voc_words)
word_to_idx = {w:i for i, w in enumerate(voc_words)}
idx_to_word = {i:w for i, w in enumerate(voc_words)}
print("vocabulary length:", '\033[32m', voc_words_size, '\033[0m', "words")
class WordDataset(Dataset):
  def __init__(self, words, context_size, test=False, by_sent=False):
    words = words.split()
    self.X = []
    self.y = []
    if by_sent == True:
        begin = 0
        sentence = []
        for i in range(len(words)):
            if words[i] == '.':
                sentence = words[begin:i]
                context_size_mod = context_size-1 if context_size<=len(sentence) else len(sentence)
                for j in range(len(sentence) - context_size_mod+1):
                    self.X.append(list(word_to_idx[sentence[k]] for k in range(j, j+context_size_mod)))
                    for k in range(j, j+context_size_mod):
                        #if k+1 == j+context_size_mod:  #### скинуть хвост в self.y 
                            #break
                        print(sentence[k], end=' ')
                    print()
                    self.y.append(word_to_idx[words[j+context_size_mod-1]])
                begin = i+1
        #print(self.X)
        for i in self.X:
            for j in i:
                print(idx_to_word[j], end=' ')
            print()
    if test==True:
        for i in range(len(words) - context_size+1):
            self.X.append(list(word_to_idx[words[j]] for j in range(i, i+context_size)))
            self.y.append(word_to_idx[words[i+context_size-1]])
    else:
        for i in range(len(words) - context_size):
          self.X.append(list(word_to_idx[words[j]] for j in range(i, i+context_size)))
          self.y.append(word_to_idx[words[i+context_size]])
    self.X = torch.tensor(self.X, dtype=torch.long)
    self.y = torch.tensor(self.y, dtype=torch.long)
      
  def __len__(self):
    return len(self.y)
  def __getitem__(self, index):
     return self.X[index], self.y[index]


ds = WordDataset(text, 4, by_sent=True)



vocabulary length:  558  words
я люблю 
я хочу любви 
мне нравится работа 
это шедевр изобразительного 
шедевр изобразительного искусства 
думайте позитивно и 
позитивно и верьте 
и верьте в 
верьте в свою 
в свою способность 
свою способность достигать 
способность достигать отличных 
достигать отличных результатов 
вы — лучший 
— лучший ответ 
лучший ответ на 
ответ на проблемы 
на проблемы , 
проблемы , которые 
, которые возникли 
которые возникли в 
возникли в понедельник 
если вы смогли 
вы смогли в 
смогли в понедельник 
в понедельник подняться 
понедельник подняться с 
подняться с постели 
с постели , 
постели , значит 
, значит вы 
значит вы супергерой 
твои утренние мысли 
утренние мысли в 
мысли в понедельник 
в понедельник задают 
понедельник задают тон 
задают тон всей 
тон всей твоей 
всей твоей неделе 
живи так чтобы 
так чтобы увидеть 
чтобы увидеть , 
увидеть , как 
, как ты 
как ты становишься 
ты становишься сильнее 
становишься сильнее и 
сильнее и живешь 
и живешь 

ValueError: expected sequence of length 2 at dim 1 (got 3)